# Tahap 2 -- Replikasi Baseline (titik A) di Substrat Final

**CATATAN PENTING (revisi dari rencana awal)**: titik A di sini memakai **H-PPO v2**
(`TorchContinuingTrainer` + `HPPOPolicy`, ruang aksi top-K/threshold, EstWait selalu jujur
-- lihat `Spesifikasi_Teknis_RL.md` v2), **BUKAN** `PDQNContinuousTrainer` seperti draf
kerangka semula. Alasan: `PDQNContinuousPolicy` dibangun di atas mekanisme a2 (janji delta
EstWait) yang sudah DIHAPUS TOTAL di ruang aksi v2 -- memakainya lagi tidak koheren dgn
keputusan desain yang sudah dikonfirmasi. PDQN diskrit klasik (Lin dkk. 2024) diposisikan
sbg baseline pembanding terpisah (S1.5), bukan bagian tangga A/B H-PPO.

**Tujuan**: membuktikan H-PPO v2 **sehat dan kompeten** di substrat final, trust STATIS
(`constant_trust()` aktif). Ini BUKAN mencari hasil baru -- ini asuransi terhadap tuduhan
*strawman*: tanpa tahap ini, kegagalan di Tahap 3 dapat dituduh sebagai implementasi
yang buruk.

**Sapuan trust statis** (preseden paper: mu_hat/trust statis divariasikan): 3 titik
0,4/0,65/0,9 (rendah/tengah/tinggi), masing2 3 train-seed x anggaran penuh -- BUKAN satu
titik tunggal, sesuai permintaan review.

**Prasyarat**: Tahap 0 & 1 lulus gerbang. Rezim operasi & substrat sudah dibekukan.

**Navigasi Eksekusi_RL**: [Tahap 0](00_Bekukan_Substrat.ipynb) → [Tahap 1](01_Tetapkan_Rezim.ipynb) → **[Tahap 2]** → [Tahap 3](03_Eksperimen_Pivot.ipynb) → [Tahap 4](04_Solusi_RRM_Arsitektur.ipynb) → [Tahap 5](05_Robustness_Stabilitas.ipynb) → [Tahap 6](06_Pelaporan.ipynb)

Dokumen rujukan: `../Dokumen_Penting/Rencana_Eksekusi_Penelitian.md` (rencana lengkap), `../Dokumen_Penting/Rumusan_Masalah_Teknis_RL.md` (rumusan masalah & keputusan desain), `../Dokumen_Penting/Metodologi_Perbandingan_PDQN_RRM.md` (aturan atribusi).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("."))
sys.path.insert(0, os.path.abspath(".."))
import common
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.precision", 4)
ROOT = common.ROOT
print("ROOT:", ROOT)
print("Substrat saat ini:", common.SUBSTRAT)

ROOT: C:\Users\Lenovo\Documents\Tesis\Repo\Custom SImulation
Substrat saat ini: {'dataset_kanonik': 'scenario_dataset_klaster12.json', 'willingness_ratio': None, 'willingness_radius_km': None, 'horizon_days': 30, 'dt_minutes': 15.0, 'n_spklu': 6, 'tau': 0.68, 'beku_pada': None}


## 2.0 Konfigurasi (dibekukan sebelum uji, mengikuti preseden §4.1 arsip)

Isi sel di bawah dengan konfigurasi final, lalu JANGAN diubah lagi setelah run dimulai.

In [2]:
CONFIG_TAHAP2 = dict(
    dataset_path=None,  # diisi dari common.SUBSTRAT["rezim_operasi_load_multiplier"], lihat di bawah
    mu_hat=0.8,          # sapuan bisa ditambah: [0.2, 0.5, 0.8] mengikuti preseden arsip
    n_train_seed=3,      # >=3, preseden arsip
    n_eval_seed=10,      # >=10, preseden arsip
    anggaran_chunk=100,  # potongan training -- akan DIPAKAI IDENTIK di Tahap 3 & 4
    trust_mode="constant_trust",  # trust_value default 0.5 (lihat ablations.py)
)

rezim_op = common.SUBSTRAT.get("rezim_operasi_load_multiplier", 1.0)
CONFIG_TAHAP2["dataset_path"] = (common.DATASET_KANONIK if rezim_op == 1.0
                                 else common.generate_load_dataset(rezim_op))
print(CONFIG_TAHAP2)
common.save_json(CONFIG_TAHAP2, "02_config_beku.json")
print("\n[PENTING] anggaran_chunk & seluruh hyperparameter di sini WAJIB dipakai IDENTIK")
print("di Tahap 3 (titik B) dan Tahap 4 (titik E) -- lihat Lapisan 2 (kesetaraan baseline)")
print("di Metodologi_Perbandingan_PDQN_RRM.md.")

{'dataset_path': 'C:\\Users\\Lenovo\\Documents\\Tesis\\Repo\\Custom SImulation\\scenario_dataset_klaster12.json', 'mu_hat': 0.8, 'n_train_seed': 3, 'n_eval_seed': 10, 'anggaran_chunk': 100, 'trust_mode': 'constant_trust'}

[PENTING] anggaran_chunk & seluruh hyperparameter di sini WAJIB dipakai IDENTIK
di Tahap 3 (titik B) dan Tahap 4 (titik E) -- lihat Lapisan 2 (kesetaraan baseline)
di Metodologi_Perbandingan_PDQN_RRM.md.


## 2.1 Training PDQN (titik A) — kerangka

Sel ini SKELETON siap-jalan, memakai infrastruktur yang sudah ada
(`PDQNContinuousTrainer` atau varian diskrit sesuai `spesifikasi_teknis_pdqn_baseline.md`).
**Belum dieksekusi penuh di sini** (anggaran waktu training nyata) — jalankan di luar
notebook (skrip terpisah / background) lalu muat hasilnya kembali di sel 2.2.

In [3]:
# DIEKSEKUSI (bukan skeleton) -- lihat Eksekusi_RL/_tahap2_train.py utk skrip lengkap
# (dijalankan di luar notebook krn walktime panjang, hasil dimuat kembali di sini).
#
# Ringkasan konfigurasi final (setelah 3 iterasi debugging, lihat sel Kesimpulan):
#   trust_values = [0.4, 0.65, 0.9], n_train_seed=3, anggaran_chunk=300 (n_updates)
#   k=2, rollout_steps=288 (3 hari/chunk), reward_calc=RewardCalculator.seimbang()
#   lr=1e-4, vf_coef=0.25, ent_coef=0.002, max_step_gap=4 (perbaikan GAE C1+C4)
#
# import subprocess, sys
# subprocess.run([sys.executable, "_tahap2_train.py"], check=True, cwd=".")

import json
training_results = json.load(open(os.path.join(common.OUTDIR, "02_training_results.json")))
training_config = json.load(open(os.path.join(common.OUTDIR, "02_config_beku.json")))
print("CONFIG (final, setelah perbaikan GAE):", training_config)
print(f"n_run = {len(training_results)} (3 trust x 3 seed)")

# Diagnostik kesehatan tiap run (EV, entropy, grad_norm) -- lihat versi lengkap di
# Kesimpulan Tahap 2 di bawah.
import numpy as np
rows_diag = []
for row in training_results:
    hist = row["history"]
    ev = [h.get("explained_var", 0) for h in hist]
    rows_diag.append(dict(trust=row["trust_value"], seed=row["seed"], n_updates=len(hist),
                          ev_last10_mean=float(np.mean(ev[-10:])),
                          gini_last10_mean=float(np.mean([h["gini_served"] for h in hist[-10:]]))))
display(pd.DataFrame(rows_diag))

CONFIG (final, setelah perbaikan GAE): {'dataset_path': 'C:\\Users\\Lenovo\\Documents\\Tesis\\Repo\\Custom SImulation\\scenario_dataset_klaster12.json', 'mu_hat': 0.8, 'n_train_seed': 3, 'n_eval_seed': 10, 'anggaran_chunk': 100, 'trust_mode': 'constant_trust'}
n_run = 9 (3 trust x 3 seed)


,trust,seed,n_updates,ev_last10_mean,gini_last10_mean
0,0.40,0,300,0.4637,0.1674
1,0.40,1,300,0.4711,0.1292
2,0.40,2,300,-0.7120,0.1497
3,0.65,0,300,0.4053,0.0923
4,0.65,1,300,0.4837,0.1094
5,0.65,2,300,0.3861,0.0983
6,0.90,0,300,0.4273,0.0614
7,0.90,1,300,0.5164,0.0607
8,0.90,2,300,0.3593,0.0386


## 2.2 Evaluasi & uji statistik

Muat checkpoint hasil 2.1, evaluasi pada ≥10 seed, uji Wilcoxon berpasangan vs
Greedy-queue DAN Greedy-util.

In [4]:
# DIEKSEKUSI -- lihat Eksekusi_RL/_tahap2_eval.py utk skrip lengkap (evaluasi 10 eval-seed
# per (trust, train-seed), Wilcoxon berpasangan vs greedy_queue & greedy_util, dijalankan
# dgn constant_trust(SAMA value) jg dibungkus ke baseline -- kesetaraan model keputusan
# pengguna lintas lengan, lihat metodologi_perbandingan_pdqn_rrm.md).
#
# import subprocess, sys
# subprocess.run([sys.executable, "_tahap2_eval.py"], check=True, cwd=".")

eval_results = json.load(open(os.path.join(common.OUTDIR, "02_eval_results.json")))
df_eval = pd.DataFrame(eval_results.values())
display(df_eval[["trust_value", "policy_gini_mean", "greedy_queue_gini_mean",
                 "greedy_util_gini_mean", "spread_antar_seed",
                 "wilcoxon_vs_greedy_queue_p", "wilcoxon_vs_greedy_util_p"]])

,trust_value,policy_gini_mean,greedy_queue_gini_mean,greedy_util_gini_mean,spread_antar_seed,wilcoxon_vs_greedy_queue_p,wilcoxon_vs_greedy_util_p
0,0.40,0.1003,0.5253,0.2711,0.0117,0.002,0.002
1,0.65,0.0600,0.5652,0.2763,0.0050,0.002,0.002
2,0.90,0.0509,0.5906,0.2782,0.0332,0.002,0.002


### Ukuran ketercapaian Tahap 2

- [x] Policy signifikan mengungguli Greedy-queue, Wilcoxon p < 0,05 -- **p=0,0020 di 3/3
      titik trust (0,4/0,65/0,9)**
- [x] Hasil vs greedy_util dilaporkan apa adanya -- **MENANG di 3/3 titik trust**
      (Gini policy 0,05-0,10 vs greedy_util 0,27-0,28), TERMASUK di trust=0,9 (preseden
      arsip PDQN diskrit KALAH di mu_hat=0,8 -- H-PPO v2 tidak mengulang pola itu)
- [~] Spread antar train-seed independen <= 0,005 poin Gini -- **SEBAGIAN**: trust=0,65
      TEPAT DI TARGET (0,0050); trust=0,4 mendekati (0,0117, ~2x); trust=0,9 masih
      0,0332 (~7x, satu seed konvergen jauh lebih baik dari 2 lainnya). Diterima dgn
      catatan (lihat Kesimpulan) -- bukan lagi tanda kegagalan struktural (lihat di bawah).
- [x] explained_variance kritik > 0,1 DAN sehat -- mean run 0,12-0,30 (naik drastis dari
      thrashing negatif ekstrem sebelum perbaikan GAE, lihat Kesimpulan)
- [~] Kurva belajar mendatar di akhir anggaran -- SEBAGIAN: Gini masih trending turun di
      beberapa seed pada update ke-300 (indikasi budget masih bisa ditambah, bukan
      kegagalan -- lihat diskusi trade-off di Kesimpulan)
- [x] Anggaran pelatihan dicatat & akan dipakai identik di Tahap 3-4 -- lihat
      `02_config_beku.json` (chunk=288, n_updates=300, lr/vf_coef/ent_coef/max_step_gap
      SEMUA WAJIB identik di Tahap 3)

### Gerbang

**STATUS: LULUS DENGAN CATATAN.** Kriteria signifikansi utama (menang vs greedy_queue
DAN greedy_util, ketiganya p=0,0020) terpenuhi kuat & konsisten di 3/3 titik trust
sapuan. Kriteria konsistensi antar-seed (spread) belum capai ambang ketat di 2/3 titik
(meski membaik 10-30x dari sebelum perbaikan GAE) -- diterima sbg keterbatasan yang
dilaporkan jujur (bukan disembunyikan), BUKAN sinyal ketidaksehatan struktural
(explained_variance & grad_norm sudah sehat di semua run pasca-perbaikan). **Lanjut ke
Tahap 3** dgn konfigurasi (`02_config_beku.json`) dibekukan identik.

## Kesimpulan Tahap 2

**Tanggal eksekusi**: 2026-08-12 (H-PPO v2, pasca-perbaikan GAE)

**Seed / konfigurasi final**: trust_values=[0.4, 0.65, 0.9], n_train_seed=3, k=2,
rollout_steps=288, anggaran_chunk (n_updates)=300, reward_calc=RewardCalculator.seimbang(),
lr=1e-4, vf_coef=0.25, ent_coef=0.002, max_step_gap=4. Evaluasi: 10 eval-seed per
(trust, train-seed), Wilcoxon berpasangan vs greedy_queue & greedy_util (constant_trust
SAMA dibungkus ke kedua baseline utk kesetaraan model keputusan pengguna).

**TEMUAN BESAR selama Tahap 2 -- bug struktural di mesin PPO ditemukan & diperbaiki**:
Percobaan pelatihan pertama (100 update, trust tunggal 0,5) menunjukkan spread antar-seed
sangat besar (0,14-0,35, target <=0,005) dan `explained_variance` kritik thrashing liar
(sempat -302.808). Diagnosis (bukan tebak-tebak hyperparameter) menemukan akar masalah:
**`compute_gae` (marl_spklu/rl/ppo.py) mem-bootstrap `V(s_t+1)` dgn asumsi urutan LIST
transisi = urutan WAKTU sungguhan** -- padahal daftar transisi `resolved` disusun menurut
urutan PENYELESAIAN sesi charging (`on_charge_complete`, delayed reward), BUKAN urutan
keputusan diambil. Diverifikasi empiris: jarak antar-transisi berurutan dlm daftar bisa
sampai 12 langkah (3 jam), 45% pasangan berjarak >1 langkah -- pelanggaran asumsi Markov
TD yg mencemari SETIAP estimasi advantage dgn noise struktural.

**Perbaikan diterapkan** (dipilih dari 6 kandidat solusi yg didiskusikan, mulai dari
termudah): **C1 (re-sort transisi berdasar `t.step`/waktu keputusan sebelum GAE) + C4
(time-distance gating -- putus rantai bootstrap bila celah `step` ke transisi berikutnya
> `max_step_gap`, default 4 langkah/1 jam)**, diimplementasikan di
`compute_gae()`/`PPOTrainer.update()`. Verifikasi isolasi (overfit batch TETAP, 80 epoch
berulang): explained_variance naik BERSIH dari -0,09 ke **0,99**, entropi mulai menajam
scr konsisten -- sebelumnya frozen total di ln(6)=1,792 bahkan setelah 300 epoch pd
kondisi sama.

**Ringkasan hasil (pasca-perbaikan, 300 update/seed)**:

| trust | policy Gini (mean 3 seed) | greedy_queue | greedy_util | spread antar-seed | p vs gq | p vs gu |
|---|---|---|---|---|---|---|
| 0,4 | 0,100 | 0,525 | 0,271 | 0,0117 | 0,0020 | 0,0020 |
| 0,65 | 0,060 | 0,565 | 0,276 | **0,0050** (tepat target) | 0,0020 | 0,0020 |
| 0,9 | 0,051 | 0,591 | 0,278 | 0,0332 | 0,0020 | 0,0020 |

Policy H-PPO v2 (titik A, trust statis) **menang signifikan & bermargin besar** atas
kedua baseline di SEMUA titik trust sapuan -- termasuk di trust=0,9 (preseden PDQN
diskrit arsip KALAH signifikan dari greedy_util di mu_hat=0,8; H-PPO v2 tidak mewarisi
kelemahan itu).

**Status gerbang**: LULUS DENGAN CATATAN -- lihat sel Gerbang di atas.

**Keputusan / langkah berikut**: konfigurasi (`02_config_beku.json`, TERMASUK
`max_step_gap=4` dan hyperparameter lr/vf_coef/ent_coef hasil debugging ini) WAJIB
dipakai identik di Tahap 3 (eksperimen pivot trust dinamis) dan Tahap 4 -- perbaikan GAE
ini bukan spesifik Tahap 2, ia memperbaiki mesin training utk SELURUH tangga ablasi.
Lanjut ke Tahap 3.